In [1]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import ResNet18, CifarDenseNet121, TinyEfficientNet
from metrics import calibration_errors, nll_loss, accuracy
import random

## Hyperparams

In [3]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
dataset = "tinyimagenet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(device="cuda")
lr = 0.1
epochs = 200
num_classes = 200
epsilon = 0.02
K = 5

## Training Utils

### Label Smoothing

In [5]:
path = f"Similarity-Aware-Label-Smoothing/scores/4_tinyimagenet_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=K, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [6]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):
    classwise_target = classwise_target.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            targets = classwise_target[y]

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(x)
                loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [7]:
classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])
print(classwise_target.shape)

tensor([0.9800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0042, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 

In [8]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [9]:
model = ResNet18(num_classes=num_classes).to(device)
model = torch.compile(model, mode="max-autotune")
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

Epoch 1/200 | Loss: 4.3509 | Test Acc: 0.1280 | Top-5 Test Acc: 0.3511


Epoch 2/200 | Loss: 3.5041 | Test Acc: 0.1985 | Top-5 Test Acc: 0.4512


Epoch 3/200 | Loss: 3.1047 | Test Acc: 0.2618 | Top-5 Test Acc: 0.5424


Epoch 4/200 | Loss: 2.8394 | Test Acc: 0.3038 | Top-5 Test Acc: 0.5891


Epoch 5/200 | Loss: 2.6625 | Test Acc: 0.3104 | Top-5 Test Acc: 0.6007


...

Epoch 196/200 | Loss: 0.1706 | Test Acc: 0.6503 | Top-5 Test Acc: 0.8344


Epoch 197/200 | Loss: 0.1708 | Test Acc: 0.6517 | Top-5 Test Acc: 0.8333


Epoch 198/200 | Loss: 0.1707 | Test Acc: 0.6507 | Top-5 Test Acc: 0.8339


Epoch 199/200 | Loss: 0.1701 | Test Acc: 0.6512 | Top-5 Test Acc: 0.8354


Epoch 200/200 | Loss: 0.1703 | Test Acc: 0.6517 | Top-5 Test Acc: 0.8346


In [10]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)

        logits_list.append(logits.float())  
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)

print("ECE:", calibration_errors(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))

test_acc = accuracy(model, test_loader)  
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


ECE: (0.041232071816921234, 0.0704105943441391)
NLL: 1.5557955503463745
Top-1 Test Acc: 0.6517 | Top-5 Test Acc: 0.8346


In [11]:
PATH = f"vae4_{'0'+str(epsilon).removeprefix("0.")}_{K}_{seed}.pth"
torch.save(model._orig_mod.state_dict(), PATH)